# Claims Assist — document extraction lab

Runs the Claims Assist extraction code (`src/app.py`, `text_extract.py`, `formatted_text_extraction.py`, `markdown_utils.py`, `docm.py`) locally.

* `local_harness.py` dispatches each extraction function by name with a JSON-style payload.
* Each payload's `path` is a local file.
* Sample documents are generated into `samples/`; drop your own files there (or use the last cell).

Sections: 1 Setup · 2 Text formats · 3 Excel · 4 Formatted PDF · 5 Password protection · 6 Extension / signature detection · 7 Archives · 8 Your own file

## 1. Setup

In [1]:
import sys, json, zipfile, tempfile
from pathlib import Path
import pandas as pd
from IPython.display import Markdown, display

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import local_harness as h            # imports src/app.py and dispatches by function name
from make_samples import build_all

S = build_all(ROOT / "samples")      # (re)generate the sample documents
print("extraction functions:", h.available_functions())
print("supported extensions    :", h.call("getSupportedFileExtensions"))
print(f"{len(S)} sample files in {ROOT/'samples'}")


def show(query, name, limit=900, **kw):
    """Run one extraction function on a sample and print the result (or the error)."""
    try:
        out = h.run(query, S[name] if name in S else name, **kw)
    except Exception as e:
        print(f"{name}: {type(e).__name__}: {e}")
        return None
    text = json.dumps(out, indent=1, ensure_ascii=False) if not isinstance(out, str) else out
    print(f"--- {query}({name}) ---")
    print(text[:limit] + (" …" if len(text) > limit else ""))
    return out

extraction functions: ['excelExtractText', 'extractEmbeddedText', 'extractFormattedPDFText', 'extractText', 'getSupportedFileExtensions', 'inspectDocumentProtection']
supported extensions    : ['.csv', '.docm', '.docx', '.htm', '.html', '.md', '.pdf', '.pptx', '.rtf', '.txt', '.xls', '.xlsb', '.xlsm', '.xlsx', '.xml']
23 sample files in /home/avenugopal/projects/claims_extract_lab/samples


## 2. Text formats
`extractText` returns a list of strings (one per page / slide / sheet).
Things to notice: html/xml come back as raw markup; docx tables (including merged cells) are joined with ` | ` and the page break becomes a second page; the Markdown sample loses its base64 signature image and has its HTML comment replaced by spaces (offsets preserved); the pptx includes table, notes and grouped text.

In [2]:
for name in ["claim_notes.txt", "claim_report.html", "adjuster_notes.md", "settlement_letter.rtf",
             "claims_register.csv", "fnol_form.docx", "fnol_form.docm", "claims_review.pptx"]:
    show("extractText", name)

--- extractText(claim_notes.txt) ---
[
 "CLAIM FILE NOTES - CL-10421 (Alder Logistics, LLC)\n2026-01-14  FNOL received by phone. Burst pipe in warehouse unit 4; stock on pallets affected.\n2026-01-16  Site inspection completed. Moisture readings elevated in 3 of 5 bays.\n2026-01-30  Indemnity payment of $10,000.00 issued to insured.\n2026-02-06  Restoration vendor invoice approved: $1,800.25.\n2026-02-12  Independent adjuster fee posted: $700.50. Total paid to date: $12,500.75.\n2026-02-20  Claim closed. No subrogation potential identified.\n"
]
--- extractText(claim_report.html) ---
[
 "<!DOCTYPE html><html><head><title>Claim CL-10435</title><style>td{padding:4px}</style></head><body><h1>Claim CL-10435 - Fire loss</h1><p>Insured: <b>Brightwater Dental</b> &middot; Policy PL-81207 &middot; Loss date 2026-02-19</p><table border='1'><tr><th>Item</th><th>Amount (USD)</th></tr><tr><td>Paid to date</td><td>36,750.40</td></tr><tr><td>Outstanding reserve</td><td>61,000.00</td></tr></table><ul

## 3. Excel
Three entry points matter: `extractText` (exactly **one** sheet allowed), `excelExtractText` (one result **per sheet**), and the engine auto-detection (`xlrd` / `openpyxl` / `pyxlsb`) based on the file signature rather than the name.

`excelExtractText` also takes an optional `method`: `"pandas"` (default, the extractor used in this notebook) or `"openpyxl"` (coordinate-aware markdown; see `02_excel_extraction_lab.ipynb`).

In [3]:
show("extractText", "loss_run.xlsx", limit=2500)         # one complex sheet: title, 3 tables, merged cell
show("extractText", "claims_workbook.xlsx")              # -> error: 4 sheets

--- extractText(loss_run.xlsx) ---
[
 "| 0   | 1                                                                                          | 2                    | 3                        | 4                   | 5                   | 6        | 7       | 8          | 9   | 10                    | 11      | 12         |\n|:----|:-------------------------------------------------------------------------------------------|:---------------------|:-------------------------|:--------------------|:--------------------|:---------|:--------|:-----------|:----|:----------------------|:--------|:-----------|\n|     |                                                                                            |                      |                          |                     |                     |          |         |            |     |                       |         |            |\n|     | Loss Run - Harbor Mutual Insurance (fictional) - amounts in USD                            |            

In [4]:
sheets = h.run("excelExtractText", S["claims_workbook.xlsx"])   # note the hidden "Internal Notes" sheet is read too
for s in sheets:
    print("sheet:", s["sheet_name"]); print(s["text"][:700]); print()

sheet: Claims
| 0 | 1 | 2 | 3 | 4 | 5 | 6 | 7 | 8 |
|:---------|:---------|:---------------------|:--------------------|:--------------------|:---------|:---------|:--------|:-----------|
| Claim ID | Policy | Insured | Date of loss | Cause | Status | Paid | Reserve | Loss ratio |
| CL-10421 | PL-77310 | Alder Logistics, LLC | 2026-01-14 00:00:00 | Water damage | Closed | 12500.75 | 0 | 0.62 |
| CL-10422 | PL-77310 | Alder Logistics, LLC | 2026-02-03 00:00:00 | Theft | Open | 4200 | 8500 | 0.45 |
| CL-10435 | PL-81207 | Brightwater Dental | 2026-02-19 00:00:00 | Fire | Open | 36750.4 | 61000 | 1.2 |
| CL-10440 | PL-90015 | Cobalt Freight | 2026-03-02 00:00:00 | Collision | Closed | 9875.1 | 0 | 0.81 |
| C

sheet: Payments
| 0 | 1 | 2 | 3 | 4 |
|:---------|:--------------------|:----------|:-------------------------|:--------|
| Claim ID | Date | Type | Payee | Amount |
| CL-10421 | 2026-01-30 00:00:00 | Indemnity | Alder Logistics, LLC | 10000 |
| CL-10421 | 2026-02-06 00:00:00 | Expen

In [5]:
# A real .xlsx that was exported with a '.xls' name - read correctly because the engine comes from the file signature
print([s["sheet_name"] for s in h.run("excelExtractText", S["loss_run_misnamed.xls"])])
print("engine for each:", {n: h.text_extract.resolve_excel_engine(Path(n).suffix, S[n].read_bytes())
                           for n in ["legacy_register.xls", "loss_run.xlsx", "loss_run_misnamed.xls"]})

['Loss Run']
engine for each: {'legacy_register.xls': 'xlrd', 'loss_run.xlsx': 'openpyxl', 'loss_run_misnamed.xls': 'openpyxl'}


### What today's Excel parser does *not* do (the motivation for the region-aware design)
In the table below note: (1) generated column numbers `0 1 2 …` replace the real headers, which become a data row; (2) the title, the side-by-side *Coverage* table and the stacked *Payment history* table are all flattened into one grid; (3) the merged `CL-10421` cell only appears on its first row; (4) loss ratios are rounded (0.6234 → 0.62); (5) the chart, text box and cell comment are ignored. Formulas here do have cached values, so totals survive — the next cell shows what happens when they don't.

In [6]:
display(Markdown(h.run("extractText", S["loss_run.xlsx"])[0]))
import openpyxl
ws = openpyxl.load_workbook(S["loss_run.xlsx"]).active
print("formula really present in the file:", ws["G14"].value)

# Same idea without cached results (what openpyxl / un-opened files look like): the total comes out blank
nc = Path(tempfile.mkdtemp()) / "no_cache.xlsx"
wb = openpyxl.Workbook(); wb.active.append(["Paid", 100]); wb.active.append(["Paid", 250]); wb.active.append(["Total", "=SUM(B1:B2)"]); wb.save(nc)
print(h.run("extractText", nc)[0])

| 0   | 1                                                                                          | 2                    | 3                        | 4                   | 5                   | 6        | 7       | 8          | 9   | 10                    | 11      | 12         |
|:----|:-------------------------------------------------------------------------------------------|:---------------------|:-------------------------|:--------------------|:--------------------|:---------|:--------|:-----------|:----|:----------------------|:--------|:-----------|
|     |                                                                                            |                      |                          |                     |                     |          |         |            |     |                       |         |            |
|     | Loss Run - Harbor Mutual Insurance (fictional) - amounts in USD                            |                      |                          |                     |                     |          |         |            |     |                       |         |            |
|     | Valuation date:                                                                            | 2026-04-30 00:00:00  |                          |                     |                     |          |         |            |     |                       |         |            |
|     |                                                                                            |                      |                          |                     |                     |          |         |            |     | Coverage details      |         |            |
|     | Claim ID                                                                                   | Insured              | Date of loss             | Cause               | Status              | Paid     | Reserve | Loss ratio |     | Coverage              | Limit   | Deductible |
|     | CL-10421                                                                                   | Alder Logistics, LLC | 2026-01-14 00:00:00      | Water damage        | Closed              | 12500.75 | 0       | 0.62       |     | Property              | 500000  | 2500       |
|     | CL-10422                                                                                   | Alder Logistics, LLC | 2026-02-03 00:00:00      | Theft               | Open                | 4200     | 8500    | 0.45       |     | General liability     | 1000000 | 5000       |
|     | CL-10435                                                                                   | Brightwater Dental   | 2026-02-19 00:00:00      | Fire                | Open                | 36750.4  | 61000   | 1.2        |     | Business interruption | 250000  | 10000      |
|     | CL-10440                                                                                   | Cobalt Freight       | 2026-03-02 00:00:00      | Collision           | Closed              | 9875.1   | 0       | 0.81       |     |                       |         |            |
|     | CL-10447                                                                                   | Cobalt Freight       | 2026-03-11 00:00:00      | Cargo loss          | Reopened            | 15300    | 7200.5  | 0.94       |     |                       |         |            |
|     | CL-10452                                                                                   | Dunmore Bakery       | 2026-03-25 00:00:00      | Equipment breakdown | Open                | 0        | 11250   | 0          |     |                       |         |            |
|     | CL-10458                                                                                   | Brightwater Dental   | 2026-04-08 00:00:00      | Liability           | Denied              | 0        | 0       | 0          |     |                       |         |            |
|     | CL-10463                                                                                   | Elm Street Clinic    | 2026-04-22 00:00:00      | Hail                | Open                | 2310.25  | 5000    | 0.33       |     |                       |         |            |
|     | Total                                                                                      |                      |                          |                     |                     | 80936.5  | 92950.5 |            |     |                       |         |            |
|     |                                                                                            |                      |                          |                     |                     |          |         |            |     |                       |         |            |
|     |                                                                                            |                      |                          |                     |                     |          |         |            |     |                       |         |            |
|     | Payment history - CL-10421                                                                 |                      |                          |                     |                     |          |         |            |     |                       |         |            |
|     | Claim                                                                                      | Type                 | Payee                    | Amount              | Date                |          |         |            |     |                       |         |            |
|     | CL-10421                                                                                   | Indemnity            | Alder Logistics, LLC     | 10000               | 2026-01-30 00:00:00 |          |         |            |     |                       |         |            |
|     |                                                                                            | Expense              | Restoration vendor       | 1800.25             | 2026-02-06 00:00:00 |          |         |            |     |                       |         |            |
|     |                                                                                            | Expense              | Independent adjuster fee | 700.5               | 2026-02-12 00:00:00 |          |         |            |     |                       |         |            |
|     |                                                                                            | Total                |                          | 12500.75            |                     |          |         |            |     |                       |         |            |
|     |                                                                                            |                      |                          |                     |                     |          |         |            |     |                       |         |            |
|     | * Reserve figures as of the valuation date. Source: claims system export (fictional data). |                      |                          |                     |                     |          |         |            |     |                       |         |            |

formula really present in the file: =SUM(G6:G13)
| 0     |   1 |
|:------|----:|
| Paid  | 100 |
| Paid  | 250 |
| Total |     |


### The openpyxl option
`excelExtractText(..., method="openpyxl")` addresses the limitations above: the real headers stay visible, side-by-side and stacked tables become separate blocks, merged cells are listed, numbers keep their precision, and every table carries Excel's column letters and row numbers so a model can cite a cell. Comments, formulas and dates stored as plain numbers are opt-in.

**For a hands-on walk through it, open [`02_excel_extraction_lab.ipynb`](02_excel_extraction_lab.ipynb)**, which runs both methods side by side on `samples/excel_edge_cases.xlsx`, a 14-tab workbook of Excel edge cases. Details: `docs/excel_to_markdown.md`.

## 4. Formatted PDF
`extractFormattedPDFText` reconstructs headings (from relative font size), **bold**/*italic*, and indentation. Every threshold in `ExtractionConfig` can be overridden per call.

In [7]:
pages = h.run("extractFormattedPDFText", S["loss_summary.pdf"])
print(f"{len(pages)} pages\n")
display(Markdown(pages[0]))
print(repr(pages[0]))

3 pages



# Q1 2026 Loss Summary

*Harbor Mutual Insurance (fictional) - valuation date 30 April 2026*

## 1. Executive summary

Paid losses for the quarter total **$80,936.50** across eight reported claims,
with outstanding case reserves of **$92,950.50** . Two claims account for
most of the movement: the Brightwater Dental fire (CL-10435) and the
reopened Cobalt Freight cargo loss (CL-10447).

## Claims at a glance

**Claim ID**
        **Insured**
                                **Cause**
                                                    **Status**
                                                                **Paid (USD)**
CL-10421
        Alder Logistics LLC
                                Water damage
                                                    Closed
                                                                12,500.75
CL-10422
        Alder Logistics LLC
                                Theft
                                                    Open
                                                                4,200.00
CL-10435
        Brightwater Dental
                                Fire
                                                    Open
                                                                36,750.40
CL-10440
        Cobalt Freight
                                Collision
                                                    Closed
                                                                9,875.10
CL-10447
        Cobalt Freight
                                Cargo loss
                                                    Reopened
                                                                15,300.00
CL-10452
        Dunmore Bakery
                                Equipment breakdown
                                                    Open
                                                                0.00
CL-10458
        Brightwater Dental
                                Liability
                                                    Denied
                                                                0.00
CL-10463
        Elm Street Clinic
                                Hail
                                                    Open
                                                                2,310.25

**Note:** *amounts exclude unallocated expense and are shown before reinsurance.*

*Confidential - Harbor Mutual (fictional) - page 1 of 3*

'# Q1 2026 Loss Summary\n\n*Harbor Mutual Insurance (fictional) - valuation date 30 April 2026*\n\n## 1. Executive summary\n\nPaid losses for the quarter total **$80,936.50** across eight reported claims,\nwith outstanding case reserves of **$92,950.50** . Two claims account for\nmost of the movement: the Brightwater Dental fire (CL-10435) and the\nreopened Cobalt Freight cargo loss (CL-10447).\n\n## Claims at a glance\n\n**Claim ID**\n        **Insured**\n                                **Cause**\n                                                    **Status**\n                                                                **Paid (USD)**\nCL-10421\n        Alder Logistics LLC\n                                Water damage\n                                                    Closed\n                                                                12,500.75\nCL-10422\n        Alder Logistics LLC\n                                Theft\n                                                    Op

In [8]:
# Tuning: with h1_ratio=3.0 the title no longer qualifies as H1 and drops to H2
print(h.run("extractFormattedPDFText", S["loss_summary.pdf"], h1_ratio=3.0)[0].splitlines()[0])
# Compare with the plain PDF extractor (note table cells: one per line in the formatted version)
print(h.run("extractEmbeddedText", S["loss_summary.pdf"])[0][:600])
print("scanned receipt (image only):", h.run("extractEmbeddedText", S["scanned_receipt.pdf"]))

## Q1 2026 Loss Summary
Q1 2026 Loss Summary
Harbor Mutual Insurance (fictional) - valuation date 30 April 2026

1. Executive summary
Paid losses for the quarter total $80,936.50 across eight reported claims,
with outstanding case reserves of $92,950.50 . Two claims account for
most of the movement: the Brightwater Dental fire (CL-10435) and the
reopened Cobalt Freight cargo loss (CL-10447).

Claims at a glance

Claim ID     Insured                Cause                  Status         Paid (USD)
CL-10421     Alder Logistics LLC        Water damage          Closed          12,500.75
CL-10422     Alder Logistics LLC   
scanned receipt (image only): ['']


## 5. Password protection
`inspectDocumentProtection` only **detects** protection (it never decrypts). Result is one of `PASSWORD_PROTECTED`, `NOT_PASSWORD_PROTECTED`, `UNKNOWN`, `NOT_APPLICABLE`.

In [9]:
rows = []
for name in ["loss_summary.pdf", "loss_summary_encrypted.pdf", "fnol_form.docx", "fnol_form_encrypted.docx",
             "loss_run.xlsx", "loss_run_encrypted.xlsx", "legacy_register.xls", "damage_photo.png", "claim_notes.txt"]:
    rows.append({"file": name,
                 "extension given": h.run("inspectDocumentProtection", S[name]),
                 "extension sniffed": h.run("inspectDocumentProtection", S[name], ext=None)})
pd.DataFrame(rows)

,file,extension given,extension sniffed
0,loss_summary.pdf,NOT_PASSWORD_PROTECTED,NOT_PASSWORD_PROTECTED
1,loss_summary_encrypted.pdf,PASSWORD_PROTECTED,PASSWORD_PROTECTED
2,fnol_form.docx,NOT_PASSWORD_PROTECTED,NOT_PASSWORD_PROTECTED
3,fnol_form_encrypted.docx,PASSWORD_PROTECTED,PASSWORD_PROTECTED
4,loss_run.xlsx,NOT_PASSWORD_PROTECTED,NOT_PASSWORD_PROTECTED
5,loss_run_encrypted.xlsx,PASSWORD_PROTECTED,PASSWORD_PROTECTED
6,legacy_register.xls,NOT_PASSWORD_PROTECTED,NOT_PASSWORD_PROTECTED
7,damage_photo.png,NOT_APPLICABLE,NOT_APPLICABLE
8,claim_notes.txt,NOT_APPLICABLE,NOT_APPLICABLE


In [10]:
# Extraction of encrypted files is not supported - see what happens:
show("extractText", "fnol_form_encrypted.docx")
print(h.run("extractEmbeddedText", S["loss_summary_encrypted.pdf"]))

ERROR:app:Could not load PDF: (.pdf) path=/home/avenugopal/projects/claims_extract_lab/samples/loss_summary_encrypted.pdf
Traceback (most recent call last):
  File "/home/avenugopal/projects/claims_extract_lab/src/app.py", line 274, in extractEmbeddedPdfText_helper
    pages = PdfExtractor().extract(file_extension, document)
  File "/home/avenugopal/projects/claims_extract_lab/src/text_extract.py", line 344, in extract
    return [page.get_text("text", sort=True).strip() for page in pdf_doc]
                                                                 ^^^^^^^
  File "/home/avenugopal/projects/claims_extract_lab/.venv/lib/python3.13/site-packages/pymupdf/__init__.py", line 2919, in __getitem__
    return self.load_page(i)
           ~~~~~~~~~~~~~~^^^
  File "/home/avenugopal/projects/claims_extract_lab/.venv/lib/python3.13/site-packages/pymupdf/__init__.py", line 5829, in load_page
    raise ValueError("document closed or encrypted")
ValueError: document closed or encrypted


fnol_form_encrypted.docx: BadZipFile: File is not a zip file
[]


## 6. Extension resolution and file-signature detection
A caller-supplied extension wins; otherwise `get_extension_from_file_signature` is used. It also finds a `%PDF` header up to 1 KB into the file.

In [11]:
rows = []
for name in ["loss_summary.pdf", "loss_summary_padded.pdf", "damage_photo.png", "fnol_form.docx", "fnol_no_extension",
             "loss_run.xlsx", "loss_run_misnamed.xls", "claims_review.pptx", "claim_package.zip",
             "legacy_register.xls", "fnol_form_encrypted.docx"]:
    rows.append({"file": name, "detected": h.app.get_extension_from_file_signature(S[name].read_bytes())})
pd.DataFrame(rows)

,file,detected
0,loss_summary.pdf,.pdf
1,loss_summary_padded.pdf,.pdf
2,damage_photo.png,.png
3,fnol_form.docx,.docx
4,fnol_no_extension,.docx
5,loss_run.xlsx,.xlsx
6,loss_run_misnamed.xls,.xlsx
7,claims_review.pptx,.pptx
8,claim_package.zip,.zip
9,legacy_register.xls,.doc


In [12]:
# No extension given -> sniffed; extractText works on a file with no extension at all
show("extractText", "fnol_no_extension", ext=None, limit=200)
print(h.app.resolve_extension("PDF", b"..."), h.app.resolve_extension(None, S["fnol_form.docx"].read_bytes()))

--- extractText(fnol_no_extension) ---
[
 "First Notice of Loss Claim CL-10421 · Policy PL-77310 · Harbor Mutual Insurance (fictional) 1. Insured and policy Insured | Alder Logistics, LLC Policy number | PL-77310 Date of loss | 14 January  …
.pdf .docx


> **Limitation.** Every OLE2 file — the legacy `legacy_register.xls`, and the *encrypted* `.docx` / `.xlsx` — is reported as **`.doc`** when no extension is supplied: the signature table has a single OLE2 entry and does not look inside. A supplied extension is trusted, so this only matters for extension-less input. `tests/test_pdf_and_detection.py` pins this behaviour.

## 7. Archives
ZIP / GZ / RAR / 7z are *recognised* by signature, but there is no extractor for them — nothing is unpacked, and `.zip` is not in the supported-extension list.

In [13]:
zp = S["claim_package.zip"]
print("contents:", zipfile.ZipFile(zp).namelist())
print("detected as:", h.app.get_extension_from_file_signature(zp.read_bytes()))
show("extractText", "claim_package.zip")

ERROR:app:Invalid File Extension: .zip


contents: ['fnol_form.docx', 'claims_register.csv', 'claim_notes.txt', 'damage_photo.png']
detected as: .zip
claim_package.zip: Exception: Invalid File Extension: .zip


## 8. Try your own file
Put a file anywhere and set `MY_FILE`. Pick any extraction function (`extractText`, `extractFormattedPDFText`, `excelExtractText`, `extractEmbeddedText`, `inspectDocumentProtection`).

In [14]:
MY_FILE = ""          # e.g. "/Users/me/Downloads/policy.pdf"  (or a name inside samples/)
FUNCTION = "extractText"
EXTRA = {}           # extra payload fields, e.g. {"method": "openpyxl", "include_comments": True} with FUNCTION = "excelExtractText"

if MY_FILE:
    out = h.run(FUNCTION, MY_FILE, ext=None, **EXTRA)    # ext=None -> detect from the file's bytes; use "auto" to trust its suffix
    print(json.dumps(out, indent=1, ensure_ascii=False)[:5000] if not isinstance(out, str) else out)
else:
    print("Set MY_FILE above and re-run this cell.")

Set MY_FILE above and re-run this cell.


---